<a href="https://colab.research.google.com/github/haticeilhan/heart-failure-clinical-analysis/blob/main/heartFailureAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install pandas numpy matplotlib seaborn scipy scikit-learn statsmodels

In [3]:
# ============================================================
# PROJECT: Heart Failure Clinical Data Analysis
# Dataset: UCI Heart Failure Clinical Records
# Goal: Explore clinical features associated with DEATH_EVENT
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, chi2_contingency

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. PROJECT SETTINGS
# ------------------------------------------------------------

RANDOM_STATE = 42

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")

print("=" * 65)
print("HEART FAILURE CLINICAL DATA ANALYSIS")
print("=" * 65)


# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

DATA_URL = (
    "https://archive.ics.uci.edu/"
    "ml/machine-learning-databases/00519/"
    "heart_failure_clinical_records_dataset.csv"
)

try:
    df = pd.read_csv(DATA_URL)
    print("\nDataset downloaded successfully.")

except Exception as error:
    raise RuntimeError(
        "Dataset could not be downloaded. Check your internet "
        "connection and try again."
    ) from error

print("\nFirst five rows:")
print(df.head().to_string())

print("\nDataset dimensions:")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nColumn names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 3. DATA QUALITY CHECKS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DATA QUALITY CHECKS")
print("=" * 65)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDescriptive statistics:")
print(df.describe().round(2).to_string())

# Save the raw dataset locally
df.to_csv(OUTPUT_DIR / "raw_dataset.csv", index=False)

# Remove exact duplicate rows if any are present
df = df.drop_duplicates().copy()

# Check the target variable
if "DEATH_EVENT" not in df.columns:
    raise ValueError("The target column DEATH_EVENT is missing.")

if not set(df["DEATH_EVENT"].dropna().unique()).issubset({0, 1}):
    raise ValueError("DEATH_EVENT must contain binary values 0 and 1.")

if df["DEATH_EVENT"].isnull().any():
    raise ValueError("The target column contains missing values.")

print("\nDataset shape after removing exact duplicates:")
print(df.shape)


# ------------------------------------------------------------
# 4. UNDERSTAND THE TARGET VARIABLE
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("OUTCOME DISTRIBUTION")
print("=" * 65)

outcome_counts = df["DEATH_EVENT"].value_counts().sort_index()

outcome_summary = pd.DataFrame({
    "Outcome": ["No recorded death event", "Death event recorded"],
    "Count": [
        int(outcome_counts.get(0, 0)),
        int(outcome_counts.get(1, 0))
    ]
})

outcome_summary["Percentage"] = (
    outcome_summary["Count"] / len(df) * 100
).round(2)

print(outcome_summary.to_string(index=False))

outcome_summary.to_csv(
    OUTPUT_DIR / "outcome_summary.csv",
    index=False
)

plt.figure(figsize=(7, 5))
sns.countplot(
    data=df,
    x="DEATH_EVENT",
    hue="DEATH_EVENT",
    legend=False
)
plt.title("Distribution of Recorded Death Events")
plt.xlabel("Death event (0 = No, 1 = Yes)")
plt.ylabel("Number of patients")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_outcome_distribution.png", dpi=300)
plt.close()


# ------------------------------------------------------------
# 5. FEATURE DESCRIPTIONS
# ------------------------------------------------------------

feature_descriptions = {
    "age": "Age in years",
    "anaemia": "Anaemia indicator",
    "creatinine_phosphokinase": "Creatinine phosphokinase level",
    "diabetes": "Diabetes indicator",
    "ejection_fraction": "Ejection fraction percentage",
    "high_blood_pressure": "High blood pressure indicator",
    "platelets": "Platelet count",
    "serum_creatinine": "Serum creatinine level",
    "serum_sodium": "Serum sodium level",
    "sex": "Sex as encoded in the dataset",
    "smoking": "Smoking indicator",
    "time": "Follow-up period in days",
    "DEATH_EVENT": "Recorded death event during follow-up"
}

print("\nFeature descriptions:")
for column in df.columns:
    print(f"{column}: {feature_descriptions.get(column, 'Clinical variable')}")


# ------------------------------------------------------------
# 6. EXPLORE NUMERICAL VARIABLES
# ------------------------------------------------------------

numerical_features = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium",
    "time"
]

# Keep only columns available in the dataset
numerical_features = [
    col for col in numerical_features if col in df.columns
]

print("\nNumerical feature summaries:")
print(df[numerical_features].describe().round(2).to_string())

df[numerical_features].describe().to_csv(
    OUTPUT_DIR / "numerical_summary.csv"
)

# Histograms
df[numerical_features].hist(
    figsize=(14, 12),
    bins=20,
    edgecolor="black"
)
plt.suptitle("Distributions of Numerical Clinical Features", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_numerical_distributions.png", dpi=300)
plt.close()


# ------------------------------------------------------------
# 7. CORRELATION ANALYSIS
# ------------------------------------------------------------

correlation_matrix = df.corr(numeric_only=True)

correlation_matrix.to_csv(
    OUTPUT_DIR / "correlation_matrix.csv"
)

plt.figure(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True
)
plt.title("Correlation Matrix of Clinical Features")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_correlation_heatmap.png", dpi=300)
plt.close()

print("\nCorrelations with DEATH_EVENT:")
print(
    correlation_matrix["DEATH_EVENT"]
    .sort_values(ascending=False)
    .round(3)
    .to_string()
)


# ------------------------------------------------------------
# 8. COMPARE NUMERICAL FEATURES BY OUTCOME
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NUMERICAL FEATURE COMPARISONS")
print("=" * 65)

comparison_results = []

for feature in numerical_features:

    group_no_event = df.loc[
        df["DEATH_EVENT"] == 0, feature
    ].dropna()

    group_event = df.loc[
        df["DEATH_EVENT"] == 1, feature
    ].dropna()

    if len(group_no_event) == 0 or len(group_event) == 0:
        continue

    # Mann-Whitney U test: a non-parametric group comparison
    statistic, p_value = mannwhitneyu(
        group_no_event,
        group_event,
        alternative="two-sided"
    )

    comparison_results.append({
        "Feature": feature,
        "No_event_median": group_no_event.median(),
        "Event_median": group_event.median(),
        "No_event_mean": group_no_event.mean(),
        "Event_mean": group_event.mean(),
        "Mann_Whitney_U": statistic,
        "p_value": p_value
    })

    print(f"\nFeature: {feature}")
    print(f"Median, no event: {group_no_event.median():.2f}")
    print(f"Median, event recorded: {group_event.median():.2f}")
    print(f"Mann-Whitney p-value: {p_value:.5f}")

comparison_df = pd.DataFrame(comparison_results)

# Apply the Benjamini-Hochberg false discovery rate correction
if not comparison_df.empty:
    from statsmodels.stats.multitest import multipletests

    comparison_df["adjusted_p_value"] = multipletests(
        comparison_df["p_value"],
        method="fdr_bh"
    )[1]

    comparison_df = comparison_df.sort_values("p_value")

    comparison_df.to_csv(
        OUTPUT_DIR / "numerical_group_comparisons.csv",
        index=False
    )

    print("\nComparison results:")
    print(comparison_df.round(4).to_string(index=False))


# ------------------------------------------------------------
# 9. BOXPLOTS: KEY CLINICAL FEATURES
# ------------------------------------------------------------

key_features = [
    feature for feature in [
        "age",
        "ejection_fraction",
        "serum_creatinine",
        "serum_sodium",
        "time"
    ]
    if feature in df.columns
]

for feature in key_features:
    plt.figure(figsize=(7, 5))

    sns.boxplot(
        data=df,
        x="DEATH_EVENT",
        y=feature,
        hue="DEATH_EVENT",
        legend=False
    )

    plt.title(f"{feature} by Recorded Death Outcome")
    plt.xlabel("Death event (0 = No, 1 = Yes)")
    plt.ylabel(feature.replace("_", " ").title())
    plt.tight_layout()

    filename = f"boxplot_{feature}.png"
    plt.savefig(OUTPUT_DIR / filename, dpi=300)
    plt.close()


# ------------------------------------------------------------
# 10. COMPARE BINARY/CATEGORICAL FEATURES
# ------------------------------------------------------------

binary_features = [
    "anaemia",
    "diabetes",
    "high_blood_pressure",
    "sex",
    "smoking"
]

binary_features = [
    feature for feature in binary_features if feature in df.columns
]

categorical_results = []

print("\n" + "=" * 65)
print("CATEGORICAL FEATURE COMPARISONS")
print("=" * 65)

for feature in binary_features:

    contingency_table = pd.crosstab(
        df[feature],
        df["DEATH_EVENT"]
    )

    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )

    categorical_results.append({
        "Feature": feature,
        "Chi_square": chi2,
        "Degrees_of_freedom": dof,
        "p_value": p_value
    })

    print(f"\nFeature: {feature}")
    print(contingency_table)
    print(f"Chi-square p-value: {p_value:.5f}")

    contingency_table.to_csv(
        OUTPUT_DIR / f"crosstab_{feature}.csv"
    )

categorical_df = pd.DataFrame(categorical_results)

if not categorical_df.empty:
    categorical_df["adjusted_p_value"] = multipletests(
        categorical_df["p_value"],
        method="fdr_bh"
    )[1]

    categorical_df.to_csv(
        OUTPUT_DIR / "categorical_group_comparisons.csv",
        index=False
    )


# ------------------------------------------------------------
# 11. PREPARE DATA FOR MACHINE LEARNING
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("LOGISTIC REGRESSION MODEL")
print("=" * 65)

# The outcome is the variable we want to predict.
X = df.drop(columns=["DEATH_EVENT"]).copy()
y = df["DEATH_EVENT"].astype(int).copy()

# All variables are numeric in this dataset.
# The imputer handles any missing predictor values.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Imputation and scaling are fitted only on training data.
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


# ------------------------------------------------------------
# 12. EVALUATE THE MODEL
# ------------------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

if y_test.nunique() == 2:
    roc_auc = roc_auc_score(y_test, y_prob)
else:
    roc_auc = np.nan

metrics = pd.DataFrame([{
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1_score": f1,
    "ROC_AUC": roc_auc
}])

print("\nModel evaluation:")
print(metrics.round(4).to_string(index=False))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

metrics.to_csv(
    OUTPUT_DIR / "model_metrics.csv",
    index=False
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted: No event", "Predicted: Event"],
    yticklabels=["Actual: No event", "Actual: Event"]
)
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_confusion_matrix.png", dpi=300)
plt.close()

# ROC curve
if y_test.nunique() == 2:
    RocCurveDisplay.from_predictions(y_test, y_prob)
    plt.title("Logistic Regression ROC Curve")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "05_roc_curve.png", dpi=300)
    plt.close()


# ------------------------------------------------------------
# 13. MODEL COEFFICIENTS
# ------------------------------------------------------------

classifier = model.named_steps["classifier"]

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Standardized_coefficient": classifier.coef_[0]
})

coefficients["Absolute_coefficient"] = (
    coefficients["Standardized_coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "Absolute_coefficient",
    ascending=False
)

coefficients.to_csv(
    OUTPUT_DIR / "model_coefficients.csv",
    index=False
)

print("\nLogistic regression coefficients:")
print(coefficients.round(4).to_string(index=False))

plt.figure(figsize=(10, 7))

plot_df = coefficients.sort_values(
    "Standardized_coefficient"
)

sns.barplot(
    data=plot_df,
    x="Standardized_coefficient",
    y="Feature",
    hue="Standardized_coefficient",
    palette="vlag",
    legend=False
)

plt.axvline(0, color="black", linewidth=1)
plt.title("Standardized Logistic Regression Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_model_coefficients.png", dpi=300)
plt.close()


# ------------------------------------------------------------
# 14. SAVE TEST SET PREDICTIONS
# ------------------------------------------------------------

predictions = pd.DataFrame({
    "Actual_DEATH_EVENT": y_test.to_numpy(),
    "Predicted_DEATH_EVENT": y_pred,
    "Predicted_probability_of_event": y_prob
})

predictions.to_csv(
    OUTPUT_DIR / "test_set_predictions.csv",
    index=False
)


# ------------------------------------------------------------
# 15. CREATE A SHORT AUTOMATED SUMMARY
# ------------------------------------------------------------

summary_lines = [
    "HEART FAILURE CLINICAL DATA ANALYSIS",
    "=" * 45,
    f"Number of patients after duplicate removal: {len(df)}",
    f"Number of clinical features: {X.shape[1]}",
    f"Recorded death-event count: {int(y.sum())}",
    f"Recorded death-event percentage: {y.mean() * 100:.2f}%",
    "",
    "Logistic Regression Test Metrics",
    f"Accuracy: {accuracy:.3f}",
    f"Precision: {precision:.3f}",
    f"Recall: {recall:.3f}",
    f"F1-score: {f1:.3f}",
    f"ROC-AUC: {roc_auc:.3f}" if not np.isnan(roc_auc)
    else "ROC-AUC: Not available",
    "",
    "Interpretation note:",
    "These results describe this dataset and model only.",
    "They do not establish causation or clinical utility.",
    "The dataset is small, so model performance may be unstable."
]

summary_text = "\n".join(summary_lines)

with open(
    OUTPUT_DIR / "project_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(summary_text)

print("\n" + "=" * 65)
print("PROJECT COMPLETED")
print("=" * 65)
print(summary_text)
print(f"\nAll output files saved in: {OUTPUT_DIR.resolve()}")

HEART FAILURE CLINICAL DATA ANALYSIS

Dataset downloaded successfully.

First five rows:
    age  anaemia  creatinine_phosphokinase  diabetes  ejection_fraction  high_blood_pressure  platelets  serum_creatinine  serum_sodium  sex  smoking  time  DEATH_EVENT
0  75.0        0                       582         0                 20                    1  265000.00               1.9           130    1        0     4            1
1  55.0        0                      7861         0                 38                    0  263358.03               1.1           136    1        0     6            1
2  65.0        0                       146         0                 20                    0  162000.00               1.3           129    1        1     7            1
3  50.0        1                       111         0                 20                    0  210000.00               1.9           137    1        0     7            1
4  65.0        1                       160         1              